In [ ]:
import os
import gzip
import json
import collections
import pandas as pd
import subprocess
import shutil
import sys
import bisect

# -----------------------
# Configuration
# -----------------------
TEST_SAMPLES_CSV = "../Data_prep/Sample_csv/processed_test_samples.csv"
THRESHOLDS_JSON = "training_results_indels/best_thresholds.json" # Pointing to Indel results
OUT_DIR = "test_results_indels"

# Paths to your blacklists
BLACKLIST_BED = "/Users/marcodupuisrodriguez/Documents/PhD/custom_blacklists/tight_clusters_under_2500bp.bed"
ARTEFACT_GENES_BED = "/Users/marcodupuisrodriguez/Documents/PhD/custom_blacklists/Artifact_Genes_Blacklist.bed"

# -----------------------
# Constants (Matched to Training Script)
# -----------------------
GENOME_SIZE = 6.4e9
R_CORD = 2e-9  

# -----------------------
# BED Lookup Logic
# -----------------------
class BedLookup:
    def __init__(self, bed_path, name):
        self.intervals = {} 
        self.name = name
        self._load_bed(bed_path)

    def _load_bed(self, bed_path):
        if not os.path.exists(bed_path):
            print(f"[WARN] BED file not found: {bed_path}. {self.name} filter will do nothing.", file=sys.stderr)
            return

        print(f"[INFO] Loading {self.name} from {bed_path}...")
        count = 0
        try:
            with open(bed_path, 'r') as f:
                for line in f:
                    if line.startswith('#') or not line.strip():
                        continue
                    parts = line.strip().split()
                    if len(parts) < 3:
                        continue
                    chrom = parts[0]
                    start, end = int(parts[1]), int(parts[2])
                    
                    if chrom not in self.intervals:
                        self.intervals[chrom] = []
                    self.intervals[chrom].append((start, end))
                    count += 1
            
            for chrom in self.intervals:
                self.intervals[chrom].sort(key=lambda x: x[0])
            print(f"[INFO] Loaded {count} regions for {self.name}.")
        except Exception as e:
            print(f"[ERROR] Error loading BED {bed_path}: {e}")

    def is_overlapped(self, chrom, pos):
        # Handle 'chr' prefix differences robustly
        regions = None
        if chrom in self.intervals:
            regions = self.intervals[chrom]
        elif f"chr{chrom}" in self.intervals:
            regions = self.intervals[f"chr{chrom}"]
        elif chrom.startswith("chr") and chrom[3:] in self.intervals:
            regions = self.intervals[chrom[3:]]
            
        if not regions:
            return False
        
        # Binary search
        query_idx = pos - 1 
        idx = bisect.bisect_right(regions, (query_idx, float('inf')))
        
        for i in range(max(0, idx - 1), -1, -1):
            start, end = regions[i]
            if end <= query_idx:
                break 
            if start <= query_idx < end:
                return True
        return False

# -----------------------
# Helper Functions
# -----------------------
def safe_float(x, default=0.0):
    try:
        return float(x)
    except (ValueError, TypeError):
        return default

def safe_int(x, default=0):
    try:
        return int(float(x))
    except (ValueError, TypeError):
        return default

def parse_info(info_str):
    info_dict = {}
    if not info_str or info_str == ".":
        return info_dict
    for item in info_str.split(";"):
        if "=" in item:
            k, v = item.split("=", 1)
            info_dict[k] = v
        else:
            info_dict[item] = "Yes"
    return info_dict

def check_dependencies():
    return shutil.which("bgzip") is not None and shutil.which("tabix") is not None

# Updated Expected Rate Formula to match Training
def get_expected_rate(sample_type, age):
    st = str(sample_type).lower()
    current_age = float(age) if pd.notna(age) else 0.0
    
    if "cord" in st:
        rate = R_CORD
    elif "pbmc" in st:
        rate = R_CORD + (0.7 * current_age) / GENOME_SIZE
    elif "t_cells" in st or "tcell" in st or "t_cell" in st:
        rate = (15.0 / GENOME_SIZE) + (1.1 * current_age) / GENOME_SIZE
    else:
        # Fallback to the original baseline
        rate = R_CORD + (2.0 * current_age) / GENOME_SIZE
        
    return rate

# -----------------------
# Filtering Logic (INDEL SPECIFIC)
# -----------------------
def apply_filters(info, alt_seq, thresholds, pon_flags): # NEW: Added alt_seq parameter
    """
    Apply INDEL-specific filters to a single variant.
    """
    n_total = safe_int(info.get("N_TOTAL", 0))
    n_total_sscs = safe_int(info.get("N_TOTAL_SSCS", -1))
    if n_total_sscs == -1: n_total_sscs = 0 
    n_alt_sscs = safe_int(info.get("N_ALT_SSCS", 0))

    indel_length = safe_int(info.get("INDEL_LENGTH", 0))
    mean_inset_qual = safe_float(info.get("MEAN_INSET_QUAL", -1.0))
    mean_del_anchor_qual = safe_float(info.get("MEAN_DEL_ANCHOR_QUAL", -1.0))
    homopolymer_len = safe_int(info.get("HOMOPOLYMER_LEN", 0))
    n_slippage_sscs = safe_int(info.get("N_SLIPPAGE_SSCS", 0))
    asxs = safe_float(info.get("AVG_ALT_ASXS", 0.0))
    mean_mapq = safe_float(info.get("MEAN_ALT_MAPQ", 0.0))
    gc_content = safe_float(info.get("GC_CONTENT", 0.5))
    dist_to_other_indel = safe_int(info.get("DIST_TO_OTHER_INDEL", 10000))
    rp_med = safe_float(info.get("RP_MED_SSCS", 75.0))
    avg_alt_n_count = safe_float(info.get("AVG_ALT_N_COUNT", 0.0))
    depth_percentile = safe_int(info.get("DEPTH_PERCENTILE", 50))
    variants_20bp = safe_int(info.get("VARIANTS_20BP", 0))
    variants_250bp = safe_int(info.get("VARIANTS_250BP", 0))
    ref_entropy = safe_float(info.get("REF_ENTROPY_40BP", 2.0))
    avg_alt_softclip = safe_float(info.get("AVG_ALT_SOFTCLIP", 0.0))
    median_dist_softclip = safe_int(info.get("MEDIAN_DIST_TO_SOFTCLIP", 1000))
    avg_alt_nm = safe_float(info.get("AVG_ALT_EXCESS_NM", 0.0))
    in_blacklist = info.get("IN_BLACKLIST", "NO")
    in_cb_pon = info.get("IN_CB_PON", "No")

    vaf_sscs = n_alt_sscs / n_total_sscs if n_total_sscs > 0 else 0.0

    # 1. Baseline Safety
    if n_total_sscs <= 0: return False, "HARD_N_TOTAL_SSCS"
    if n_alt_sscs <= 1: return False, "HARD_N_ALT_SSCS" 
    
    # NEW: Filter out insertions that contain 'N' in the inserted sequence
    if indel_length > 0 and 'N' in str(alt_seq).upper(): return False, "HARD_N_IN_INSERTION"

    if in_cb_pon == "Yes" or in_blacklist == "YES": return False, "FAIL_CORD_PON"

    # 2. Depth & Shared Quality
    if n_total < thresholds["n_total_min"]: return False, "FAIL_MIN_DEPTH"
    if depth_percentile > thresholds["depth_percentile_max"]: return False, "FAIL_DEPTH_PERCENTILE"
    if mean_mapq < thresholds["mean_mapq_min"]: return False, "FAIL_MAPQ"
    if asxs <= thresholds["asxs_min"]: return False, "FAIL_ASXS"

    # 3. Split Quality Logic (Ins vs Del)
    is_ins = indel_length > 0
    is_del = indel_length < 0
    
    if is_ins and mean_inset_qual < thresholds.get('mean_inset_qual_min', 30): return False, "FAIL_INSET_QUAL"
    elif is_del and mean_del_anchor_qual < thresholds.get('mean_del_anchor_qual_min', 30): return False, "FAIL_DEL_ANCHOR_QUAL"
    elif not is_ins and not is_del: return False, "FAIL_INDEL_LENGTH_ZERO"

    # 4. Homopolymer Logic
    abs_len = abs(indel_length)
    if abs_len == 1 and homopolymer_len > thresholds["hp_max_1bp"]: return False, "FAIL_HP_1BP"
    elif abs_len > 1 and homopolymer_len > thresholds["hp_max_long"]: return False, "FAIL_HP_LONG"
    if n_slippage_sscs > thresholds["max_slippage_reads"]: return False, "FAIL_SLIPPAGE"

    # 5. Alignment & Context
    if variants_20bp > thresholds["variants_20bp_max"]: return False, "FAIL_VARIANTS_20BP"
    if variants_250bp > thresholds["variants_250bp_max"]: return False, "FAIL_VARIANTS_250BP"
    if ref_entropy < thresholds["ref_entropy_min"]: return False, "FAIL_REF_ENTROPY"
    if avg_alt_softclip > thresholds["avg_alt_softclip_max"]: return False, "FAIL_AVG_ALT_SOFTCLIP"
    if median_dist_softclip < thresholds["dist_to_softclip_min"]: return False, "FAIL_DIST_TO_SOFTCLIP"
    if avg_alt_nm > thresholds["avg_alt_nm_max"]: return False, "FAIL_AVG_ALT_NM"

    # 6. Additional Filters
    if gc_content < thresholds["gc_min"] or gc_content > thresholds["gc_max"]: return False, "FAIL_GC_CONTENT"
    if dist_to_other_indel < thresholds["min_dist_to_indel"]: return False, "FAIL_DIST_TO_OTHER_INDEL"
    if rp_med < thresholds["min_rp_med"] or rp_med > thresholds["max_rp_med"]: return False, "FAIL_RP_MED"
    if avg_alt_n_count > thresholds["max_n_count"]: return False, "FAIL_N_COUNT"

    # 7. Germline
    if vaf_sscs > thresholds["germline_vaf_max"]: return False, "FAIL_GERMLINE_VAF"

    return True, None

# -----------------------
# Main Execution
# -----------------------
if __name__ == "__main__":
    os.makedirs(OUT_DIR, exist_ok=True)
    has_htslib = check_dependencies()

    print("[INFO] Initializing Blacklists...")
    cluster_lookup = BedLookup(BLACKLIST_BED, "Cluster Blacklist")
    artefact_lookup = BedLookup(ARTEFACT_GENES_BED, "Artefact Gene Blacklist")

    if not os.path.exists(THRESHOLDS_JSON):
        print(f"[ERROR] Thresholds file not found: {THRESHOLDS_JSON}")
        sys.exit(1)

    with open(THRESHOLDS_JSON) as f:
        data = json.load(f)
        thresholds = data["thresholds"]
        pon_flags = data.get("pon_flags", {})
    print("[INFO] Thresholds loaded.")

    if not os.path.exists(TEST_SAMPLES_CSV):
        print(f"[ERROR] Test samples CSV not found: {TEST_SAMPLES_CSV}")
        sys.exit(1)

    df_samples = pd.read_csv(TEST_SAMPLES_CSV)
    print(f"[INFO] Processing {len(df_samples)} test samples...")

    results_data = []
    all_fail_reasons = []
    
    # Track overall Indel Ratios across the test cohort
    cohort_n_ins = 0
    cohort_n_del = 0
    cohort_n_1bp = 0
    cohort_n_long = 0

    for idx, s in df_samples.iterrows():
        print(f"  -> {s.sample_id}", end="\r")
        
        input_basename = os.path.basename(s.vcf_path).replace(".vcf", "").replace(".gz", "")
        out_vcf_path = os.path.join(OUT_DIR, f"{input_basename}.filtered.vcf")
        
        if not os.path.exists(s.vcf_path):
            print(f"\n[WARN] VCF not found: {s.vcf_path}")
            continue

        openf = gzip.open if s.vcf_path.endswith(".gz") else open
        n_pass = 0
        
        try:
            with openf(s.vcf_path, "rt") as fin, open(out_vcf_path, "w") as fout:
                for line in fin:
                    if line.startswith("#"):
                        fout.write(line)
                        continue
                    
                    fields = line.strip().split("\t")
                    if len(fields) < 8: continue
                    
                    chrom, pos = fields[0], int(fields[1])
                    alt_seq = fields[4] # NEW: Extract the ALT sequence
                    
                    if cluster_lookup.is_overlapped(chrom, pos):
                        all_fail_reasons.append("FAIL_CLUSTER_BLACKLIST")
                        continue 
                    
                    if artefact_lookup.is_overlapped(chrom, pos):
                        all_fail_reasons.append("FAIL_ARTEFACT_BLACKLIST")
                        continue 
                    
                    info = parse_info(fields[7])
                    
                    # NEW: Pass alt_seq to apply_filters
                    passed, reason = apply_filters(info, alt_seq, thresholds, pon_flags) 
                    
                    if passed:
                        n_pass += 1
                        fout.write(line)
                        
                        # Tally Indel Metrics for passed variants
                        indel_length = safe_int(info.get("INDEL_LENGTH", 0))
                        if indel_length > 0:
                            cohort_n_ins += 1
                        elif indel_length < 0:
                            cohort_n_del += 1
                            
                        if abs(indel_length) == 1:
                            cohort_n_1bp += 1
                        elif abs(indel_length) > 1:
                            cohort_n_long += 1
                    else:
                        all_fail_reasons.append(reason)
        except Exception as e:
            print(f"\n[ERROR] Failed on {s.sample_id}: {e}")
            continue

        if has_htslib:
            try:
                subprocess.run(["bgzip", "-f", out_vcf_path], check=True, stderr=subprocess.DEVNULL)
                subprocess.run(["tabix", "-p", "vcf", out_vcf_path + ".gz"], check=True, stderr=subprocess.DEVNULL)
            except:
                pass
        
        sample_type = s.sample_type if "sample_type" in s else "T_cells"
        age = s.age if "age" in s else 0.0
        bases = float(s.bases)
        
        exp_rate = get_expected_rate(sample_type, age)
        exp_count = exp_rate * bases
        
        results_data.append({
            "sample_id": s.sample_id,
            "sample_type": sample_type,
            "age": age,
            "bases": bases,
            "observed": n_pass,
            "expected": exp_count
        })

    print("\n[INFO] Filtering complete.")

    res_df = pd.DataFrame(results_data)
    res_df.to_csv(os.path.join(OUT_DIR, "test_summary.csv"), index=False)
    
    reason_counts = collections.Counter(all_fail_reasons)
    fail_df = pd.DataFrame(reason_counts.items(), columns=["Reason", "Count"]).sort_values("Count", ascending=False)
    fail_df.to_csv(os.path.join(OUT_DIR, "filter_failure_counts.csv"), index=False)

    # --- Cohort Indel Ratios ---
    ins_del_ratio = cohort_n_ins / cohort_n_del if cohort_n_del > 0 else 0.0
    bp1_long_ratio = cohort_n_1bp / cohort_n_long if cohort_n_long > 0 else 0.0

    print("-" * 30)
    print("FINAL TEST RESULTS")
    print("-" * 30)
    print(f"Results saved to: {OUT_DIR}")
    print(f"Total passing variants: {sum(r['observed'] for r in results_data)}")
    print("-" * 30)
    print("TEST COHORT INDEL METRICS")
    print("-" * 30)
    print(f"Insertions: {cohort_n_ins} | Deletions: {cohort_n_del}")
    print(f"INS / DEL Ratio:       {ins_del_ratio:.3f}")
    print(f"1bp Indels: {cohort_n_1bp} | >1bp Indels: {cohort_n_long}")
    print(f"1bp / >1bp Ratio:      {bp1_long_ratio:.3f}")
    print("-" * 30)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from matplotlib.offsetbox import AnchoredText

# -----------------------
# Configuration
# -----------------------
# Path to the CSV generated by the test filtering step
CSV_PATH = "test_results_indels/test_summary.csv"
OUTPUT_PNG = "test_results_indels/test_observed_vs_expected_plot.png"
EPS = 1e-15

# -----------------------
# Metric Calculation
# -----------------------
def poisson_deviance(y_arr, lam_arr):
    """
    Calculates Poisson Deviance between observed (y) and expected (lambda).
    Formula: 2 * sum( y * log(y / lambda) - (y - lambda) )
    """
    y = np.asarray(y_arr, dtype=float)
    lam = np.asarray(lam_arr, dtype=float) + EPS
    
    with np.errstate(divide='ignore', invalid='ignore'):
        log_term = np.log(y / lam)
        term1 = np.where(y > 0, y * log_term, 0.0)
    
    dev = 2.0 * (term1 - (y - lam))
    return float(np.sum(dev))

# Check if file exists
if not os.path.exists(CSV_PATH):
    print(f"[ERROR] Could not find {CSV_PATH}. Please run the test filtering cell first.")
else:
    # Load table
    df = pd.read_csv(CSV_PATH)

    # -----------------------
    # Data Processing & Aggregation
    # -----------------------
    # Identify cord blood samples
    is_cord = df['sample_type'].str.lower().str.contains('cord')
    
    df_cord = df[is_cord]
    df_other = df[~is_cord]

    # Extract raw counts for non-cord samples
    ages_list = list(df_other["age"].values)
    obs_list = list(df_other["observed"].values)
    exp_list = list(df_other["expected"].values)
    bases_list = list(df_other["bases"].values)

    # Aggregate cord samples if they exist
    if not df_cord.empty:
        total_cord_obs = df_cord["observed"].sum()
        total_cord_exp = df_cord["expected"].sum()
        total_cord_bases = df_cord["bases"].sum()
        
        # Append the aggregated cord blood point
        ages_list.append(0.0) # Cord blood is age 0
        obs_list.append(total_cord_obs)
        exp_list.append(total_cord_exp)
        bases_list.append(total_cord_bases)

    # Convert back to numpy arrays
    ages = np.array(ages_list)
    observed = np.array(obs_list)
    expected = np.array(exp_list)
    bases = np.array(bases_list)
    
    # Calculate Indel rates per base
    obs_rate = observed / bases
    exp_rate = expected / bases

    # Calculate Deviance on the aggregated data
    deviance = poisson_deviance(observed, expected)
    total_obs = np.sum(observed)
    total_exp = np.sum(expected)

    # -----------------------
    # Plotting
    # -----------------------
    fig, ax = plt.subplots(figsize=(8, 6))

    # Plot Observed (Blue circles)
    ax.scatter(ages, obs_rate, label="Observed", color="#1f77b4", alpha=0.8, marker="o", s=60, edgecolors='k', linewidth=0.5)

    # Plot Expected (Orange crosses)
    ax.scatter(ages, exp_rate, label="Expected", color="#ff7f0e", alpha=0.8, marker="x", s=60, linewidth=2)

    # Styling
    ax.set_xlabel("Age (years)", fontsize=12)
    ax.set_ylabel("Indel rate (Indels per base)", fontsize=12)
    
    # Handle X-limits (pad slightly around min/max age)
    valid_ages = ages[~np.isnan(ages)]
    if len(valid_ages) > 0:
        ax.set_xlim(min(valid_ages) - 2, max(valid_ages) + 5)
    else:
        ax.set_xlim(-2, 100)

    # Hardcoded Y-limits (Same as training plot for consistency)
    ax.set_ylim(0, 2e-8)
    
    ax.set_title("Observed vs Expected Indel Rate (Test Set)", fontsize=14)
    ax.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.5)
    
    # Add Stats Box (Deviance)
    stats_text = (f"Poisson Deviance: {deviance:.2f}\n"
                  f"Total Obs: {int(total_obs)}\n"
                  f"Total Exp: {total_exp:.1f}")
    
    at = AnchoredText(stats_text, prop=dict(size=10), frameon=True, loc='upper right')
    at.patch.set_boxstyle("round,pad=0.2,rounding_size=0.2")
    at.patch.set_alpha(0.8)
    ax.add_artist(at)

    # Legend
    ax.legend(loc='upper left', frameon=True, fontsize=10)

    plt.tight_layout()
    
    # Save and Show
    plt.savefig(OUTPUT_PNG, dpi=150)
    print(f"[INFO] Poisson Deviance on Test Set: {deviance:.4f}")
    print(f"[INFO] Plot saved to: {OUTPUT_PNG}")
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# -----------------------
# Configuration
# -----------------------
TRAIN_CSV = "training_results_indels/observed_expected.csv"
TEST_CSV = "test_results_indels/test_summary.csv"
OUTPUT_PNG = "./cord_blood_snv_comparison.png"
REFERENCE_RATE = 2e-9

def get_cord_blood_rate(csv_path):
    """
    Loads CSV, filters for age == 0 (Cord Blood), 
    and calculates aggregate SNV rate.
    """
    if not os.path.exists(csv_path):
        print(f"[WARNING] File not found: {csv_path}")
        return 0.0, 0  # Return 0 rate and 0 count
        
    df = pd.read_csv(csv_path)
    
    # Filter for Cord Blood (Age = 0)
    # Using a small epsilon or direct comparison depending on data precision
    cord_data = df[np.isclose(df["age"], 0, atol=1e-5)]
    
    if cord_data.empty:
        print(f"[WARNING] No Cord Blood (age=0) data found in {csv_path}")
        return 0.0, 0
    
    total_obs = cord_data["observed"].sum()
    total_bases = cord_data["bases"].sum()
    
    if total_bases == 0:
        return 0.0, 0
        
    return total_obs / total_bases, total_obs

# -----------------------
# Main Execution
# -----------------------
# 1. Get Rates
train_rate, train_obs_count = get_cord_blood_rate(TRAIN_CSV)
test_rate, test_obs_count = get_cord_blood_rate(TEST_CSV)

# 2. Prepare Data for Plotting
labels = ['Train Cord', 'Test Cord', 'Reference']
rates = [train_rate, test_rate, REFERENCE_RATE]
colors = ['#cccccc', '#1f77b4', '#d62728']  # Grey (Train), Blue (Test), Red (Ref)

# 3. Plotting
fig, ax = plt.subplots(figsize=(6, 6))

# Create Bars
bars = ax.bar(labels, rates, color=colors, edgecolor='k', alpha=0.8)

# 4. Styling
ax.set_ylabel("SNV Rate (SNVs per base)", fontsize=12)
ax.set_title("Cord Blood SNV Rate Comparison", fontsize=14)
ax.grid(axis='y', linestyle='--', alpha=0.5)

# Add value labels on top of bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2e}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# Adjust Y-limit to fit labels
ax.set_ylim(0, max(rates) * 1.2)

plt.tight_layout()

# 5. Save and Show
plt.savefig(OUTPUT_PNG, dpi=150)
print(f"[INFO] Train Cord Rate: {train_rate:.2e} (Obs: {int(train_obs_count)})")
print(f"[INFO] Test Cord Rate:  {test_rate:.2e} (Obs: {int(test_obs_count)})")
print(f"[INFO] Reference Rate:  {REFERENCE_RATE:.2e}")
print(f"[INFO] Plot saved to: {OUTPUT_PNG}")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from matplotlib.offsetbox import AnchoredText

# -----------------------
# Configuration
# -----------------------
# Paths to your results
TEST_CSV_PATH = "test_results_indels/test_summary.csv"           
TRAIN_CSV_PATH = "training_results_indels/observed_expected.csv" 

OUTPUT_PNG = "./test_vs_train_overlay_plot.png"
EPS = 1e-15

# -----------------------
# Helper Functions
# -----------------------
def poisson_deviance(y_arr, lam_arr):
    """
    Calculates Poisson Deviance between observed (y) and expected (lambda).
    Formula: 2 * sum( y * log(y / lambda) - (y - lambda) )
    """
    y = np.asarray(y_arr, dtype=float)
    lam = np.asarray(lam_arr, dtype=float) + EPS
    
    with np.errstate(divide='ignore', invalid='ignore'):
        log_term = np.log(y / lam)
        term1 = np.where(y > 0, y * log_term, 0.0)
    
    dev = 2.0 * (term1 - (y - lam))
    return float(np.sum(dev))

def aggregate_cord_blood(df):
    """
    Identifies Cord Blood samples, aggregates them into a single data point, 
    and returns a combined dataframe.
    """
    if df.empty:
        return df
        
    is_cord = df['sample_type'].str.lower().str.contains('cord')
    df_cord = df[is_cord]
    df_other = df[~is_cord].copy()
    
    if not df_cord.empty:
        cord_row = pd.DataFrame({
            'age': [0.0],
            'observed': [df_cord['observed'].sum()],
            'expected': [df_cord['expected'].sum()],
            'bases': [df_cord['bases'].sum()],
            'sample_type': ['Cord_Aggregated']
        })
        df_other = pd.concat([df_other, cord_row], ignore_index=True)
        
    return df_other

# -----------------------
# Data Loading & Processing
# -----------------------
if not os.path.exists(TEST_CSV_PATH):
    print(f"[ERROR] Could not find Test CSV: {TEST_CSV_PATH}")
    test_df = pd.DataFrame(columns=["age", "observed", "expected", "bases", "sample_type"])
else:
    test_df = pd.read_csv(TEST_CSV_PATH)
    test_df = aggregate_cord_blood(test_df)

if not os.path.exists(TRAIN_CSV_PATH):
    print(f"[WARN] Could not find Training CSV: {TRAIN_CSV_PATH}. Plotting Test only.")
    train_df = pd.DataFrame(columns=["age", "observed", "expected", "bases", "sample_type"])
else:
    train_df = pd.read_csv(TRAIN_CSV_PATH)
    train_df = aggregate_cord_blood(train_df)

# Calculate Deviance for Test Set (on aggregated data)
if not test_df.empty:
    test_obs = test_df["observed"].values
    test_exp = test_df["expected"].values
    test_deviance = poisson_deviance(test_obs, test_exp)
    test_total_obs = np.sum(test_obs)
    test_total_exp = np.sum(test_exp)
else:
    test_deviance = 0
    test_total_obs = 0
    test_total_exp = 0

# -----------------------
# Plotting
# -----------------------
fig, ax = plt.subplots(figsize=(10, 7))

# --- 1. Plot Training Data (Background / Grey) ---
if not train_df.empty:
    train_ages = train_df["age"].values
    train_rates_obs = train_df["observed"].values / (train_df["bases"].values + EPS)
    train_rates_exp = train_df["expected"].values / (train_df["bases"].values + EPS)
    
    # Plot Training Observed (Grey Circles)
    ax.scatter(train_ages, train_rates_obs, 
               label="Training Observed", 
               color="silver", 
               alpha=0.6, 
               marker="o", 
               s=50, 
               zorder=1) 

    # Plot Training Expected (Grey Crosses)
    ax.scatter(train_ages, train_rates_exp, 
               label="Training Expected", 
               color="grey", 
               alpha=0.6, 
               marker="x", 
               s=50, 
               zorder=1) 

# --- 2. Plot Test Data (Foreground / Colored) ---
if not test_df.empty:
    test_ages = test_df["age"].values
    test_rates_obs = test_df["observed"].values / (test_df["bases"].values + EPS)
    test_rates_exp = test_df["expected"].values / (test_df["bases"].values + EPS)

    # Plot Test Observed (Blue Circles)
    ax.scatter(test_ages, test_rates_obs, 
               label="Test Observed", 
               color="#1f77b4", 
               alpha=0.9, 
               marker="o", 
               s=70, 
               edgecolors='k', 
               linewidth=0.8,
               zorder=2)

    # Plot Test Expected (Orange Crosses)
    ax.scatter(test_ages, test_rates_exp, 
               label="Test Expected", 
               color="#ff7f0e", 
               alpha=0.9, 
               marker="x", 
               s=70, 
               linewidth=2.5,
               zorder=2)

# -----------------------
# Styling & Stats
# -----------------------
ax.set_xlabel("Age (years)", fontsize=12)
ax.set_ylabel("Indel rate (Indels per base)", fontsize=12) # Updated Label

# Handle X-limits
all_ages = np.concatenate([
    test_df["age"].values if not test_df.empty else [],
    train_df["age"].values if not train_df.empty else []
])
valid_ages = all_ages[~np.isnan(all_ages)]

if len(valid_ages) > 0:
    ax.set_xlim(min(valid_ages) - 2, max(valid_ages) + 5)
else:
    ax.set_xlim(-2, 100)

# Y-limits (Updated to match previous plots)
ax.set_ylim(0, 2e-8)

ax.set_title("Observed vs Expected Indel Rate: Test Set Overlay", fontsize=14) # Updated Title
ax.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.5)

# Add Stats Box (Test Data Only)
stats_text = (f"TEST SET METRICS\n"
              f"Poisson Deviance: {test_deviance:.2f}\n"
              f"Total Obs: {int(test_total_obs)}\n"
              f"Total Exp: {test_total_exp:.1f}")

at = AnchoredText(stats_text, prop=dict(size=10), frameon=True, loc='upper right')
at.patch.set_boxstyle("round,pad=0.2,rounding_size=0.2")
at.patch.set_alpha(0.9)
at.patch.set_facecolor("white")
ax.add_artist(at)

# Legend
ax.legend(loc='upper left', frameon=True, fontsize=10)

plt.tight_layout()

# Save and Show
plt.savefig(OUTPUT_PNG, dpi=150)
print(f"[INFO] Test Deviance: {test_deviance:.4f}")
print(f"[INFO] Plot saved to: {OUTPUT_PNG}")
plt.show()